# 03 - Disposal intent from disclosure text

Reported data cannot tell you what management plans to do. That lives in prose:
strategic plan commentary, NPE reduction targets, results-call language. Nobody
reads it across fifty banks by hand, in five languages, every quarter.

This is where the LLM earns its place -- not as a summariser, but as a
classifier over text that would otherwise go unread.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)


In [2]:
from src import intent as I

print("DEEPSEEK_API_KEY present:", I.api_available())
print("Model:", I.MODEL)
I.SIGNAL_LADDER

DEEPSEEK_API_KEY present: True
Model: deepseek-chat


{'none': 0,
 'denial': 0,
 'monitoring': 25,
 'exploring': 50,
 'target': 75,
 'programme': 100}

## The schema

Intent is graded on a ladder, weakest to strongest. The ladder matters more than
the labels: an announced, sized programme is a different object from "we keep
all options under review".

A horizon multiplier then discounts anything without a near-term date, because
a 2030 target is not an origination signal.

In [3]:
print(I.HORIZON_MULTIPLIER)
corpus = I.load_corpus()
print(f"\nCorpus entries: {len(corpus)}")
print(I.build_prompt(corpus[0]))

{'near': 1.0, 'medium': 0.75, 'long': 0.5, 'unknown': 0.6}

Corpus entries: 13
You are reading disclosure text from a European bank to assess whether it intends to sell non-performing loan portfolios.

Bank: UNICREDIT, SOCIETA' PER AZIONI
Source: 2025-Annual-Reports-and-Accounts.pdf (2025)

TEXT:
"""The aforementioned strategies concerning impaired loans include: • an effective internal restructuring activity, aiming at bringing fully back to bonis the portfolio thanks to qualified resources with specific skills dedicated to the management of loans classified as unlikely to pay; • proactive portfolio management through judicial and extra-judicial procedures managed by internal Workout professionals or assigned to external agencies specialised in credit recovery; • optimization of recovery performance thanks to formalised partnerships with specialised servicers; • disposal of impaired loans as further strategy for internal recovery both for individual positions and for portfolios of imp

## Three rules that make this usable

1. **Every signal carries its source sentence and a URL.** An extracted score
   with no quote behind it cannot be used in a credit process and would rightly
   be ignored.
2. **The model never sees or produces a number.** It classifies text into a
   fixed schema; all arithmetic happens in Python afterwards.
3. **Quotes are verified character-for-character against the source.** If the
   model returns evidence that does not appear in the input text, the evidence
   is dropped and the signal is flagged. This is the answer to the first
   question any credit professional asks about LLM output.

In [4]:
signals = I.run_extraction()   # live LLM where a key is present
for s in signals:
    print(f"{s.bank[:40]:<40} {s.signal:<11} {s.horizon:<8} "
          f"verified={str(s.verified):<5} score={s.score():>5.0f}")

UNICREDIT, SOCIETA' PER AZIONI           programme   unknown  verified=True  score=   60
UNICREDIT, SOCIETA' PER AZIONI           programme   unknown  verified=True  score=   60
UNICREDIT, SOCIETA' PER AZIONI           programme   unknown  verified=True  score=   60
Banco Bilbao Vizcaya Argentaria, S.A.    none        unknown  verified=True  score=    0
Deutsche Pfandbriefbank AG               monitoring  unknown  verified=True  score=   15
Deutsche Pfandbriefbank AG               target      near     verified=True  score=   75
Deutsche Pfandbriefbank AG               monitoring  unknown  verified=True  score=   15
Deutsche Pfandbriefbank AG               monitoring  unknown  verified=True  score=   15
Deutsche Pfandbriefbank AG               none        unknown  verified=True  score=    0
Deutsche Pfandbriefbank AG               none        unknown  verified=True  score=    0
Banca Monte dei Paschi di Siena S.p.A.   none        unknown  verified=True  score=    0
Banca Monte dei Pasch

## Provenance gating

`IntentSignal.score()` returns **0** for anything not marked
`VERIFIED_VERBATIM`, and a quote earns that mark only by being *located in the
PDF it claims to come from* (`scripts/promote_candidates.py` re-opens the
cached document and matches the text). Verification is mechanical and
reproducible rather than someone asserting they checked.

Every entry in the corpus has passed that check. Earlier builds carried
labelled placeholder text to exercise the pipeline before real documents were
reachable; it has been removed now that the pipeline demonstrably runs on real
filings, because invented sentences attached to real bank names are a liability
in a public repository even when correctly flagged and scored at zero.

A bank absent from the corpus is reported as **intent unconfirmed**, never as
silent -- absence of data is not evidence of absence.

In [5]:
from collections import Counter
print(Counter(s.provenance for s in signals))

print("\nScored vs held:")
for s_ in signals:
    mark = "VERIFIED" if s_.verified else "held"
    print(f"  {s_.bank[:34]:<34} {s_.signal:<11} {mark:<9} {s_.score():>5.0f}")

Counter({'VERIFIED_VERBATIM': 13})

Scored vs held:
  UNICREDIT, SOCIETA' PER AZIONI     programme   VERIFIED     60
  UNICREDIT, SOCIETA' PER AZIONI     programme   VERIFIED     60
  UNICREDIT, SOCIETA' PER AZIONI     programme   VERIFIED     60
  Banco Bilbao Vizcaya Argentaria, S none        VERIFIED      0
  Deutsche Pfandbriefbank AG         monitoring  VERIFIED     15
  Deutsche Pfandbriefbank AG         target      VERIFIED     75
  Deutsche Pfandbriefbank AG         monitoring  VERIFIED     15
  Deutsche Pfandbriefbank AG         monitoring  VERIFIED     15
  Deutsche Pfandbriefbank AG         none        VERIFIED      0
  Deutsche Pfandbriefbank AG         none        VERIFIED      0
  Banca Monte dei Paschi di Siena S. none        VERIFIED      0
  Banca Monte dei Paschi di Siena S. none        VERIFIED      0
  Banca Monte dei Paschi di Siena S. none        VERIFIED      0


## The case that proves the gate is working

BBVA is the useful result, not UniCredit.

Its quote is **verified** -- genuinely located in the 2025 Annual Report -- and
the classifier still returned `none`, because the passage discusses an NPL ratio
moving rather than any intention to sell. The pipeline found text, confirmed it
was real, and then declined to manufacture a signal from it.

A screen that only ever finds signal is not a screen. Being able to point at a
verified input that correctly scored zero is what makes the non-zero scores
worth anything.

To extend coverage: `scripts/harvest_disclosures.py`, review the candidates,
then `scripts/promote_candidates.py --apply`.